# 042-optimised — Convert the second ("optimised") selection's records to JSON

Identical to nb `042` except that it reads the conversion lists written by `040-…-optimised`
(in `cfg["proc_data"]["gm_selection_optimised"]`). The one-off 2026-09-30 ESM reconversion switches
and dry run of nb `042` are left out: that reconversion is done, and the converter now always uses
the channel map and network code.

**Output folder is SHARED with the original selection**: `cfg["proc_data"]["gm_records"]`. A JSON
is named after the physical record (`{record_identifier}__{component}.json`), so a record selected
by both selections is the same file. Existing JSONs are **skipped, never overwritten**
(`overwrite=False`), so only the records new to the optimised selection are converted and the
files the original MSA runs used are unchanged. The spectra file likewise only gains entries
(`update_spectra` reuses every spectrum whose JSON is unchanged).

**Upstream** (from `040-…-optimised`, in the optimised folder):
- `esm_records_to_convert.csv`, `ngasub_records_to_convert.csv`.

Also reads the raw archives (`cfg["raw_data"]["esm_hdf5_folder"]`, `cfg["raw_data"]["ngasub_folder"]`)
and the NGA-Sub reference flatfiles in `cfg["raw_data"]["gm_flatfiles"]`.

**Run order** — run top to bottom after `040-…-optimised`. Records still missing from the raw
archives are reported as skipped.

In [ ]:
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.data_handling.convert_esm_records_to_json import (
    convert_esm_records,
)
from phd_project.scripts.data_handling.convert_ngasub_records_to_json import (
    convert_ngasub_records,
)

cfg = config.load_config()

## Paths

In [ ]:
# Source record archives (network share - external resource)
ESM_SRC = cfg["raw_data"]["esm_hdf5_folder"]
NGASUB_SRC = cfg["raw_data"]["ngasub_folder"]

# Conversion lists of the OPTIMISED selection (from 040-optimised)
esm_selection_csv = cfg["proc_data"]["gm_selection_optimised"] / "esm_records_to_convert.csv"
ngasub_selection_csv = cfg["proc_data"]["gm_selection_optimised"] / "ngasub_records_to_convert.csv"

# NGA-Sub reference flatfiles
ngasub_filename_map = cfg["raw_data"]["gm_flatfiles"] / "NGASub_SA_H1_H2_filenamemap.csv"
ngasub_metadata = cfg["raw_data"]["gm_flatfiles"] / "NGASub_Metadata_SA_rotD50.csv"

# Output folder for the converted JSON records: SHARED with the original selection (see the
# header). Existing JSONs are skipped, so nothing the original selection uses is rewritten.
output_dir = cfg["proc_data"]["gm_records"]

## Convert ESM records

In [ ]:
# overwrite=False: only records without a JSON are converted. Never set this to True here:
# the folder is shared with the original selection (see the header).
n_esm, esm_skipped = convert_esm_records(
    ESM_SRC, esm_selection_csv, output_dir, overwrite=False)

## Convert NGA-Sub records

In [ ]:
n_ngasub, ngasub_skipped = convert_ngasub_records(
    NGASUB_SRC, ngasub_selection_csv, output_dir,
    filename_map=ngasub_filename_map, metadata=ngasub_metadata)

## Records that could not be converted

Any selected records that were skipped (e.g. missing source file) are listed
below for inspection.

In [ ]:
esm_skipped_df = pd.DataFrame(esm_skipped, columns=["record_identifier", "component", "reason"])
ngasub_skipped_df = pd.DataFrame(ngasub_skipped, columns=["record_identifier", "component", "reason"])

print(f"ESM skipped: {len(esm_skipped_df)} | NGASub skipped: {len(ngasub_skipped_df)}")
display(esm_skipped_df)
display(ngasub_skipped_df)

## Record spectra

`compute_record_spectra.update_spectra` keeps `07_gm_records/spectra/acc_spectra.pickle` in line with the
record JSONs in `output_dir` (only the `*.json` files directly in it, so `BACKUP_DIR` is ignored). Spectra
and JSONs are matched by name (record_id = JSON file stem):
1. The existing spectra file is read, if there is one. Its spectra are reused only if they were made with
   the same method, damping and period grid; otherwise all are recomputed, and the reason is printed.
2. Spectra without a JSON are **removed**, and their tags are printed.
3. JSONs **without a spectrum** are computed and added.
4. Spectra whose JSON was rewritten after they were computed (e.g. by `RECONVERT_ESM = True`) are
   recomputed, even though the name is the same, because each spectrum stores its JSON's modification time.

`SPECTRA_METHOD` (see the docstring of `compute_record_spectra.psa_spectrum`):
* `"fft"` (default): the record is first resampled to dt ≤ 0.002 s by band-limited (FFT) interpolation.
  Matches the NGA-Sub flatfile spectra to ~0.1 % at every record dt.
* `"zoh"`: the original scheme, on the record's own dt. It is biased low for coarse records (dt ≥ 0.02 s,
  some NGA-Sub records): ~13 % at T = 0.2 s and ~60 % at T = 0.05 s when dt = 0.05 s (nb 043 §9).

The spectra file in place on 2026-10-01 was made with `"zoh"`, so the first run with `"fft"` recomputes
every record.

In [ ]:
from psutil import cpu_count
from phd_project.scripts.compute_record_spectra import update_spectra

SPECTRA_METHOD = "fft"      # "fft" (default) or "zoh" (the original scheme), see the markdown above
N_CORES = max((cpu_count(logical=False) or 1) - 3, 1)               # worker processes

# the file nb 043 reads (timehistory_spectra_verification.TH_SPECTRA_FP)
spectra_fp = output_dir / "spectra" / "acc_spectra.pickle"

spectra = update_spectra(output_dir, spectra_fp, method=SPECTRA_METHOD, n_cores=N_CORES)

failed = spectra.attrs["failed"]
print(f"records whose JSON could not be used: {len(failed)}")
for rid, err in failed.items():
    print(f"  {rid}: {err}")
spectra.iloc[:5, :8]